# Intermediate 04 — Fine-Grained Authorization with OPA, Cedar & OpenFGA

We implement the same **Claims Assistant** scenario using three authorization mental models.

```text
Alice -> Claims Agent -> Task claim-483
                         |
                         +-> Claim 483
                         +-> claim.read
                         +-> claim.update
                         +-> payment.create (approval + <= $500)
```

The executable notebook uses local Python evaluators so no infrastructure is required. It also writes/ships real Rego, Cedar and OpenFGA model artifacts for use with their native tooling.


In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
import json, time, uuid

def decision(engine, allow, reason, request):
    return {
        "decision_id": str(uuid.uuid4()),
        "engine": engine,
        "allow": bool(allow),
        "reason": reason,
        "subject": request["user"]["id"],
        "actor": request["agent"]["id"],
        "action": request["action"],
        "resource": request["resource"]["id"],
        "task": request["task"]["id"],
    }


## 1 — Scenario and trusted facts

In [ ]:
BASE = {
    "user": {"id":"alice","department":"claims"},
    "agent": {"id":"claims-agent","risk_tier":1,"workload_trusted":True},
    "task": {"id":"task:483","active":True,"claim_id":"claim:483"},
    "resource": {"id":"claim:483","type":"claim","owner":"alice","fraud_hold":False},
    "action":"claim.read",
    "context":{"risk_score":10,"approved":False,"amount":0},
}
print(json.dumps(BASE,indent=2))


## 2 — Reference policy: authority intersection

In [ ]:
AGENT_MAX = {
    "claims-agent":{"claim.read","claim.update","payment.create"}
}
USER_RIGHTS = {
    "alice":{"claim.read","claim.update","payment.create"}
}
TASK_RIGHTS = {
    "task:483":{"claim.read","claim.update","payment.create"}
}

def reference_authorize(r):
    action=r["action"]
    if not r["agent"]["workload_trusted"]:
        return decision("reference",False,"untrusted workload",r)
    if not r["task"]["active"]:
        return decision("reference",False,"task inactive",r)
    if r["resource"]["id"] != r["task"]["claim_id"]:
        return decision("reference",False,"resource outside task",r)
    if action not in USER_RIGHTS.get(r["user"]["id"],set()):
        return decision("reference",False,"user lacks authority",r)
    if action not in AGENT_MAX.get(r["agent"]["id"],set()):
        return decision("reference",False,"agent lacks authority",r)
    if action not in TASK_RIGHTS.get(r["task"]["id"],set()):
        return decision("reference",False,"task lacks authority",r)
    if r["resource"].get("fraud_hold") and action != "claim.read":
        return decision("reference",False,"fraud hold",r)
    if action=="payment.create":
        if not r["context"]["approved"]:
            return decision("reference",False,"approval required",r)
        if r["context"]["amount"] > 500:
            return decision("reference",False,"autonomous limit exceeded",r)
    return decision("reference",True,"all constraints satisfied",r)

reference_authorize(BASE)


## 3 — Negative tests

In [ ]:
import copy

def changed(**kwargs):
    r=copy.deepcopy(BASE)
    for path,value in kwargs.items():
        node=r
        parts=path.split(".")
        for p in parts[:-1]:
            node=node[p]
        node[parts[-1]]=value
    return r

tests=[
 ("wrong resource", changed(**{"resource.id":"claim:999"}), False),
 ("expired task", changed(**{"task.active":False}), False),
 ("untrusted workload", changed(**{"agent.workload_trusted":False}), False),
]
for name,r,expected in tests:
    got=reference_authorize(r)["allow"]
    print(name, got)
    assert got==expected


## 4 — OPA/Rego mental model

In [ ]:
def opa_style_authorize(r):
    # Mirrors policies/opa/claims.rego.
    allow=False
    reason="default deny"
    same_claim=r["resource"]["id"]==r["task"]["claim_id"]
    base=(r["user"]["id"]=="alice" and
          r["agent"]["id"]=="claims-agent" and
          r["task"]["active"] and same_claim)
    if base and r["action"]=="claim.read":
        allow,reason=True,"Rego read rule matched"
    elif base and r["action"]=="claim.update" and r["context"]["risk_score"]<50:
        allow,reason=True,"Rego update rule matched"
    elif (base and r["action"]=="payment.create" and
          r["context"]["approved"] and r["context"]["amount"]<=500):
        allow,reason=True,"Rego payment rule matched"
    return decision("OPA/Rego",allow,reason,r)

opa_style_authorize(BASE)


The real Rego policy is included at `policies/opa/claims.rego`, with tests at `claims_test.rego`.

Run locally with OPA:

```bash
opa test policies/opa -v
```

OPA's native test framework makes policy-as-code CI straightforward.


## 5 — OPA contextual payment decision

In [ ]:
pay=changed(**{
    "action":"payment.create",
    "context.approved":True,
    "context.amount":450
})
print(opa_style_authorize(pay))

too_much=changed(**{
    "action":"payment.create",
    "context.approved":True,
    "context.amount":900
})
print(opa_style_authorize(too_much))


## 6 — Cedar PARC

In [ ]:
def cedar_request(r):
    return {
      "principal": f'Agent::"{r["agent"]["id"]}"',
      "action": f'Action::"{r["action"]}"',
      "resource": f'Claim::"{r["resource"]["id"]}"',
      "context":{
        "onBehalfOf":r["user"]["id"],
        "taskActive":r["task"]["active"],
        "taskClaimId":r["task"]["claim_id"],
        "amount":r["context"]["amount"],
        "approved":r["context"]["approved"],
      }
    }
print(json.dumps(cedar_request(BASE),indent=2))


## 7 — Cedar permit + forbid semantics

In [ ]:
def cedar_style_authorize(r):
    same=(r["resource"]["id"]==r["task"]["claim_id"])
    permit=False
    if (r["agent"]["id"]=="claims-agent" and
        r["user"]["id"]=="alice" and
        r["task"]["active"] and same):
        if r["action"]=="claim.read":
            permit=True
        if (r["action"]=="payment.create" and
            r["context"]["approved"] and
            r["context"]["amount"]<=500):
            permit=True

    forbid=(r["action"]=="payment.create" and r["context"]["amount"]>500)

    if forbid:
        return decision("Cedar",False,"matching forbid overrides permit",r)
    if permit:
        return decision("Cedar",True,"matching permit and no forbid",r)
    return decision("Cedar",False,"no permit",r)

print(cedar_style_authorize(pay))
print(cedar_style_authorize(too_much))


The real Cedar policy is included at `policies/cedar/claims.cedar`.

Notice the security property:

```text
forbid match -> DENY
otherwise permit match -> ALLOW
otherwise -> DENY
```

Cedar schemas should be used in production to validate entity/action/context shapes.


## 8 — OpenFGA relationship graph

In [ ]:
TUPLES={
 ("agent:claims-agent","can_act_on_behalf_of","user:alice"),
 ("agent:claims-agent","assignee","task:483"),
 ("user:alice","requester","task:483"),
 ("user:alice","owner","claim:483"),
 ("agent:claims-agent","viewer","claim:483"),
 ("agent:claims-agent","editor","claim:483"),
 ("agent:claims-agent","can_call","tool:claim-read"),
 ("agent:claims-agent","can_call","tool:claim-update"),
}

def has(subject,relation,obj):
    return (subject,relation,obj) in TUPLES

print(has("agent:claims-agent","can_act_on_behalf_of","user:alice"))
print(has("agent:claims-agent","viewer","claim:483"))


## 9 — OpenFGA-style checks

In [ ]:
def fga_style_authorize(r):
    agent=f'agent:{r["agent"]["id"]}'
    user=f'user:{r["user"]["id"]}'
    resource=r["resource"]["id"]
    task=r["task"]["id"]

    if not has(agent,"can_act_on_behalf_of",user):
        return decision("OpenFGA",False,"no delegation relationship",r)
    if not has(agent,"assignee",task):
        return decision("OpenFGA",False,"agent not assigned to task",r)
    if r["action"]=="claim.read" and has(agent,"viewer",resource):
        return decision("OpenFGA",True,"viewer relationship",r)
    if r["action"]=="claim.update" and has(agent,"editor",resource):
        return decision("OpenFGA",True,"editor relationship",r)
    return decision("OpenFGA",False,"relationship not found",r)

print(fga_style_authorize(BASE))


## 10 — Revocation is a relationship mutation

In [ ]:
revoked=set(TUPLES)
revoked.discard(("agent:claims-agent","viewer","claim:483"))

def has_revoked(subject,relation,obj):
    return (subject,relation,obj) in revoked

print("before:",has("agent:claims-agent","viewer","claim:483"))
print("after :",has_revoked("agent:claims-agent","viewer","claim:483"))


## 11 — Tool authorization and resource authorization

In [ ]:
def may_call_tool(agent,tool):
    return has(f"agent:{agent}","can_call",f"tool:{tool}")

def secure_tool_read(r):
    tool_ok=may_call_tool(r["agent"]["id"],"claim-read")
    resource_ok=has(f'agent:{r["agent"]["id"]}',"viewer",r["resource"]["id"])
    return tool_ok and resource_ok

print("tool + resource:",secure_tool_read(BASE))


A tool permission must not silently become wildcard permission over every object that tool can touch.

## 12 — Authorization-aware RAG

In [ ]:
DOCS=[
 {"id":"doc:public","claim":"claim:483","text":"General claim procedure"},
 {"id":"doc:483-note","claim":"claim:483","text":"Alice claim investigation note"},
 {"id":"doc:999-medical","claim":"claim:999","text":"Bob confidential medical record"},
]
RAG_VIEW={
 ("user:alice","doc:public"),
 ("user:alice","doc:483-note"),
}

def retrieve(_query):
    # Simulates vector candidates.
    return DOCS

def auth_filter(user,docs):
    return [d for d in docs if (f"user:{user}",d["id"]) in RAG_VIEW]

candidates=retrieve("claim medical details")
authorized=auth_filter("alice",candidates)
print("retrieved:",[d["id"] for d in candidates])
print("sent to LLM:",[d["id"] for d in authorized])
assert "doc:999-medical" not in [d["id"] for d in authorized]


## 13 — Hybrid OpenFGA + contextual policy

In [ ]:
def hybrid_authorize(r):
    # Relationship layer
    rel=fga_style_authorize(r)
    if not rel["allow"]:
        return decision("Hybrid",False,"relationship layer denied",r)

    # Contextual guardrail layer
    if r["context"]["risk_score"]>=50:
        return decision("Hybrid",False,"risk policy denied",r)
    if r["resource"].get("fraud_hold") and r["action"]!="claim.read":
        return decision("Hybrid",False,"fraud hold",r)

    return decision("Hybrid",True,"relationship + contextual policy allowed",r)

print(hybrid_authorize(BASE))


## 14 — Decision log

In [ ]:
requests=[
 BASE,
 changed(**{"task.active":False}),
 changed(**{"resource.id":"claim:999"}),
]
logs=[reference_authorize(r) for r in requests]
print(json.dumps(logs,indent=2))


## 15 — Adversarial regression suite

In [ ]:
cases=[
 ("prompt claims approval", changed(**{"action":"payment.create","context.approved":False,"context.amount":100}), False),
 ("amount escalation", changed(**{"action":"payment.create","context.approved":True,"context.amount":501}), False),
 ("wrong claim", changed(**{"resource.id":"claim:999"}), False),
 ("trusted read", BASE, True),
]
for name,r,expected in cases:
    result=reference_authorize(r)
    print(name,result["allow"],result["reason"])
    assert result["allow"]==expected


## 16 — Exercise: model user + agent intersection in OpenFGA

Require:

```text
Alice can view claim:483
AND
claims-agent can act for Alice
AND
claims-agent assigned to task:483
```

Do not copy Alice's entire permission set onto the agent.


## 17 — Exercise: contextual task grant

Create an ephemeral task authorization for:

```text
agent:claims-agent
action: claim.update
resource: claim:483
expires: 30 minutes
```

Compare how you would represent this in:

- OPA input/data;
- Cedar context/entities;
- OpenFGA conditions/contextual tuples.


## 18 — Exercise: MCP

Tools:

```text
claim.search
claim.read
claim.update
payment.create
```

Implement:

1. tool-level authorization;
2. target-resource authorization;
3. approval for `payment.create`;
4. independent revocation of the agent.


## 19 — Exercise: RAG at scale

The retriever returns 100 candidate chunks.

Implement:

```text
over-fetch -> batch authorization -> authorized top-k -> LLM
```

Then compare with pre-filtering by authorized document IDs.

Measure:

```text
latency
number of PDP calls
number of candidates
security boundary
```


## 20 — Exercise: native engines

Run the shipped artifacts against the real systems:

### OPA

```bash
opa test policies/opa -v
opa eval -d policies/opa/claims.rego -i input.json 'data.agent.claims.allow'
```

### Cedar

Use the Cedar CLI/SDK or Cedar playground with:

```text
policies/cedar/claims.cedar
```

Add a complete schema and entity set.

### OpenFGA

Start OpenFGA, load:

```text
policies/openfga/model.fga
```

then create tuples and execute Check/ListObjects queries.


## 21 — Architecture challenge

Design an enterprise agent authorization plane where:

```text
OpenFGA -> relationship graph
OPA or Cedar -> contextual guardrails
OAuth -> transportable delegated authority
SPIFFE -> workload identity
PEP -> MCP/API/RAG boundaries
```

Explain:

- source of truth for each fact;
- caching;
- revocation;
- failure mode;
- decision evidence;
- policy/model versioning.


## 22 — Review questions

1. Why is authentication insufficient for tool execution?
2. What are PEP and PDP?
3. Why should authorization default deny?
4. How do RBAC, ABAC and ReBAC differ?
5. Why should an agent be a first-class principal?
6. Why is effective authority often an intersection?
7. What kinds of policies are natural in Rego?
8. What does Cedar PARC mean?
9. How does Cedar `forbid` interact with `permit`?
10. Why are Cedar schemas valuable?
11. What does OpenFGA store as relationship tuples?
12. Why does ReBAC fit graph-shaped agent workflows?
13. Why is tool authorization separate from resource authorization?
14. Where should RAG authorization occur?
15. What are pre-filter and post-retrieval authorization trade-offs?
16. When would you choose OPA?
17. When would you choose Cedar?
18. When would you choose OpenFGA?
19. Why might a hybrid architecture be best?
20. Which authorization inputs must never come only from the LLM?

# Next course

## Intermediate 05 — Dynamic Authorization & Continuous Access Evaluation
